# Aula 6 - Treino da rede neuronal
## Losses (Slides 3-5)

In [1]:
import numpy as np
from si.neural_networks.losses import LossFunction, MeanSquaredError, BinaryCrossEntropy

# 1. a classe base é abstrata: não se pode criar um objeto dela
try:
    LossFunction()
except TypeError as e:
    print("LossFunction() ->", e)

# 2. gradient check: a derivative tem de bater certo com a derivada numérica da loss
#    derivada numérica de cada valor: (L(p + eps) - L(p - eps)) / (2 * eps)
def derivada_numerica(loss_fn, y_true, y_pred, eps=1e-6):
    numerica = np.zeros_like(y_pred)
    for i in range(y_pred.shape[0]):
        for j in range(y_pred.shape[1]):
            p_mais, p_menos = y_pred.copy(), y_pred.copy()
            p_mais[i, j] += eps
            p_menos[i, j] -= eps
            numerica[i, j] = (loss_fn.loss(y_true, p_mais) - loss_fn.loss(y_true, p_menos)) / (2 * eps)
    return numerica

np.random.seed(0)
y_true = np.random.randint(0, 2, size=(5, 1)).astype(float)  # rótulos 0/1
y_pred = np.random.uniform(0.05, 0.95, size=(5, 1))         # probabilidades longe de 0 e de 1

for loss_fn in [MeanSquaredError(), BinaryCrossEntropy()]:
    analitica = loss_fn.derivative(y_true, y_pred)
    numerica = derivada_numerica(loss_fn, y_true, y_pred)
    print(f"{type(loss_fn).__name__}: loss = {loss_fn.loss(y_true, y_pred):.4f} | "
          f"diferença máx. = {np.abs(analitica - numerica).max():.1e}")

# 3. a BCE castiga muito uma previsão confiante e errada
bce = BinaryCrossEntropy()
y = np.array([[1.0]])
print("y = 1, p = 0.9  ->", round(bce.loss(y, np.array([[0.9]])), 3))
print("y = 1, p = 0.01 ->", round(bce.loss(y, np.array([[0.01]])), 3))
print("y = 1, p = 0    ->", round(bce.loss(y, np.array([[0.0]])), 3), "(sem o clip dava inf)")

LossFunction() -> Can't instantiate abstract class LossFunction with abstract methods derivative, loss
MeanSquaredError: loss = 0.2969 | diferença máx. = 5.3e-11
BinaryCrossEntropy: loss = 4.0772 | diferença máx. = 5.1e-10
y = 1, p = 0.9  -> 0.105
y = 1, p = 0.01 -> 4.605
y = 1, p = 0    -> 34.539 (sem o clip dava inf)


**Conclusão (losses):**
- A `derivative` de cada loss bate certo com a derivada numérica (diferença ~1e-10): é mesmo a derivada da `loss`. É este valor que entra no backward pela última camada.
- MSE para regressão; BCE para classificação binária, com sigmoide na última camada.
- A BCE castiga muito os erros confiantes: com y = 1, p = 0.9 dá 0.105, mas p = 0.01 dá 4.6.
- O `np.clip` evita log(0) = -inf: com p = 0 a loss fica 34.5 em vez de inf.


## Optimizers (Slides 6-8)


In [2]:
from si.neural_networks.optimizers import Optimizer, SGD

# classe base é abstrata
try:
    Optimizer(0.01)
except TypeError as e:
    print("Optimizer() ->", e)

# com momentum = 0, o SGD é o gradient descent normal: w - α * gradiente
w = np.array([[1.0, -2.0], [0.5, 3.0]])
grad = np.array([[0.2, -0.4], [1.0, 0.0]])
sgd = SGD(learning_rate=0.1)
print("Igual a w - α·grad:", np.allclose(sgd.update(w, grad), w - 0.1 * grad))
print("retained_gradient criado com a shape de w:", sgd.retained_gradient.shape)

# efeito do momentum: minimizar f(w) = w², cujo mínimo é w = 0 (gradiente = 2w)
# com α = 1.05 o passo é grande demais: o gradient descent salta por cima do mínimo
# e cada salto é maior do que o anterior (zigue-zague a crescer)
for beta in [0.0, 0.9]:
    opt = SGD(learning_rate=1.05, momentum=beta)
    w = np.array([[1.0]])
    valores = []
    for passo in range(30):
        w = opt.update(w, 2 * w)
        valores.append(round(float(w[0, 0]), 3))
    print(f"momentum = {beta}: primeiros passos {valores[:6]} ... passo 30: {valores[-1]}")
    # maior distância ao mínimo (|w|) em cada bloco de 10 passos: se cresce, diverge; se encolhe, converge
    amplitudes = [max(abs(v) for v in valores[i:i + 10]) for i in (0, 10, 20)]
    print(f"maior |w| nos passos 1-10, 11-20, 21-30: {amplitudes}")

Optimizer() -> Can't instantiate abstract class Optimizer with abstract method update
Igual a w - α·grad: True
retained_gradient criado com a shape de w: (2, 2)
momentum = 0.0: primeiros passos [-1.1, 1.21, -1.331, 1.464, -1.611, 1.772] ... passo 30: 17.449
maior |w| nos passos 1-10, 11-20, 21-30: [2.594, 6.727, 17.449]
momentum = 0.9: primeiros passos [0.79, 0.435, 0.024, -0.35, -0.614, -0.723] ... passo 30: -0.03
maior |w| nos passos 1-10, 11-20, 21-30: [0.79, 0.508, 0.273]


**Conclusão (optimizers):**
- Com `momentum = 0`, o `SGD` faz exatamente w − α·gradiente (gradient descent normal).
- O `retained_gradient` só é criado no primeiro `update`, com a shape dos parâmetros. Por isso, no `initialize` da `DenseLayer`, os pesos e os biases têm cada um a sua cópia do optimizer (`copy.deepcopy`).
- Em f(w) = w² com α = 1.05, sem momentum, cada passo salta por cima do mínimo e o salto cresce (w chega a 17.4 ao passo 30): diverge.
- Com momentum 0.9 também passa por cima do mínimo (chega a −0.72), mas cada oscilação é mais pequena do que a anterior (maior |w| por bloco de 10 passos: 0.79 → 0.51 → 0.27, contra 2.6 → 6.7 → 17.4 sem momentum). A média dos gradientes trava o zigue-zague e w aproxima-se do mínimo (−0.03 ao passo 30).

## NeuralNetwork (slides(9-13))


In [3]:
import numpy as np
from si.metrics.accuracy import accuracy
from si.io.csv_file import read_csv
from si.model_selection.split import stratified_train_test_split
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.layers import DenseLayer
from si.neural_networks.activation import ReLUActivation, SigmoidActivation
from si.neural_networks.optimizers import SGD
from si.neural_networks.losses import BinaryCrossEntropy

# accuracy: rótulos 1D como antes, e agora também probabilidades (n, 1) e (n, k)
y = np.array([0, 1, 1])
print("rótulos 1D:      ", round(accuracy(y, np.array([0, 1, 0])), 3))
print("prob. classe 1:  ", round(accuracy(y, np.array([[0.2], [0.9], [0.4]])), 3))
print("prob. por classe:", round(accuracy(y, np.array([[0.7, 0.3], [0.1, 0.9], [0.6, 0.4]])), 3))

# rede no breast-bin (9 features, classificação binária)
data = read_csv('../datasets/breast_bin/breast-bin.csv', sep=',', features=True, label=True)
train, test = stratified_train_test_split(data, test_size=0.3, random_state=42)

np.random.seed(42)
net = NeuralNetwork(epochs=20, batch_size=16, optimizer=SGD, learning_rate=0.01,
                    loss=BinaryCrossEntropy, metric=accuracy)
net.add(DenseLayer(4, (9,)))   # só a 1.ª camada precisa do input_shape
net.add(ReLUActivation())
net.add(DenseLayer(1))
net.add(SigmoidActivation())

# o add ligou as camadas: o input de cada uma é o output da anterior
for layer in net.layers:
    print(f"{layer.layer_name():18} input {layer.input_shape()} -> output {layer.output_shape()} | parâmetros: {layer.parameters()}")

net.fit(train)
for epoch in [1, 10, 20]:
    h = net.history[epoch]
    print(f"época {epoch:2}: loss = {h['loss']:.2f} | accuracy = {h['metric']:.3f}")

print("previsões:", net.predict(test).shape, "| accuracy no teste:", round(net.score(test), 3))

rótulos 1D:       0.667
prob. classe 1:   0.667
prob. por classe: 0.667
DenseLayer         input (9,) -> output (4,) | parâmetros: 40
ReLUActivation     input (4,) -> output (4,) | parâmetros: 0
DenseLayer         input (4,) -> output (1,) | parâmetros: 5
SigmoidActivation  input (1,) -> output (1,) | parâmetros: 0
época  1: loss = 235.63 | accuracy = 0.820
época 10: loss = 66.36 | accuracy = 0.955
época 20: loss = 58.34 | accuracy = 0.965
previsões: (209, 1) | accuracy no teste: 0.967


**Conclusão (NeuralNetwork):**
- O `add` liga as camadas sozinho: só a 1.ª precisa do `input_shape`. Parâmetros: 9·4 + 4 = 40 e 4·1 + 1 = 5.
- Em cada mini-batch: forward → derivada da loss → backward. Os pesos mudam depois de cada mini-batch (com batch_size = 16, são ~31 atualizações por época), e não só uma vez por época.
- A rede aprende: no treino, a loss desce de 235.6 para 58.3 e a accuracy sobe de 0.82 para 0.965. A BCE é uma soma, por isso o valor depende do número de amostras (≈ 0.12 por amostra no fim).
- No teste, com dados que a rede nunca viu, a accuracy é 0.967.
- A `accuracy` passou a aceitar as probabilidades da rede: (n, 1) arredonda e (n, k) usa argmax. Os rótulos 1D ficam iguais, por isso os modelos das outras aulas não mudam.

## XOR (Slide 14)

In [4]:
from si.data.dataset import Dataset
from si.neural_networks.losses import MeanSquaredError

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([0, 1, 1, 0], dtype=float)
ds = Dataset(X, y)

np.random.seed(1)
net = NeuralNetwork(epochs=5000, batch_size=4, optimizer=SGD, learning_rate=0.5,
                    loss=MeanSquaredError, metric=accuracy)
net.add(DenseLayer(4, (2,)))
net.add(SigmoidActivation())
net.add(DenseLayer(1))
net.add(SigmoidActivation())
net.fit(ds)
print(net.predict(ds).round(2))   # ~[0.05, 0.95, 0.95, 0.07]

[[0.05]
 [0.95]
 [0.95]
 [0.07]]


In [5]:
def treinar_xor(seed=1, learning_rate=0.5, sigmoide_no_meio=True, epochs=5000):
    """
    Treina a rede do slide 14 com uma variação e devolve a rede treinada.
    """
    np.random.seed(seed)
    net = NeuralNetwork(epochs=epochs, batch_size=4, optimizer=SGD, learning_rate=learning_rate,
                        loss=MeanSquaredError, metric=accuracy)
    net.add(DenseLayer(4, (2,)))
    if sigmoide_no_meio:
        net.add(SigmoidActivation())
    net.add(DenseLayer(1))
    net.add(SigmoidActivation())
    net.fit(ds)
    return net


variacoes = {
    "base (slide)": dict(),
    "sem a sigmoide do meio": dict(sigmoide_no_meio=False),
    "learning_rate = 0.01": dict(learning_rate=0.01),
    "np.random.seed(0)": dict(seed=0),
}

for nome, kwargs in variacoes.items():
    net = treinar_xor(**kwargs)
    # loss em algumas épocas, para ver em que altura (e se) a rede começa a aprender
    perdas = [round(float(net.history[e]['loss']), 3) for e in [1, 1000, 2000, 5000]]
    print(f"{nome:23} previsões {net.predict(ds).round(2).ravel()} | accuracy {net.score(ds):.2f} "
          f"| loss nas épocas 1/1000/2000/5000: {perdas}")

base (slide)            previsões [0.05 0.95 0.95 0.07] | accuracy 1.00 | loss nas épocas 1/1000/2000/5000: [0.25, 0.249, 0.162, 0.003]
sem a sigmoide do meio  previsões [0.5 0.5 0.5 0.5] | accuracy 0.50 | loss nas épocas 1/1000/2000/5000: [0.25, 0.25, 0.25, 0.25]
learning_rate = 0.01    previsões [0.5 0.5 0.5 0.5] | accuracy 0.50 | loss nas épocas 1/1000/2000/5000: [0.25, 0.25, 0.25, 0.25]
np.random.seed(0)       previsões [0.5 0.5 0.5 0.5] | accuracy 0.50 | loss nas épocas 1/1000/2000/5000: [0.257, 0.25, 0.25, 0.25]


In [6]:
# primeira época em que a rede acerta os 4 casos (None se nunca acertar)
def epoca_accuracy_1(net):
    return next((epoca for epoca, h in net.history.items() if h['metric'] == 1.0), None)

mais_epocas = {
    "base (slide)": dict(),
    "sem a sigmoide do meio": dict(sigmoide_no_meio=False, epochs=30000),
    "np.random.seed(0)": dict(seed=0, epochs=30000),
    "learning_rate = 0.01": dict(learning_rate=0.01, epochs=150000),  # demora uns segundos
}

for nome, kwargs in mais_epocas.items():
    net = treinar_xor(**kwargs)
    print(f"{nome:23} {net.epochs:6} épocas | accuracy 1 na época {epoca_accuracy_1(net)} "
          f"| previsões {net.predict(ds).round(2).ravel()}")

base (slide)              5000 épocas | accuracy 1 na época 2207 | previsões [0.05 0.95 0.95 0.07]
sem a sigmoide do meio   30000 épocas | accuracy 1 na época None | previsões [0.5 0.5 0.5 0.5]
np.random.seed(0)        30000 épocas | accuracy 1 na época 10962 | previsões [0.02 0.98 0.98 0.02]
learning_rate = 0.01    150000 épocas | accuracy 1 na época 110165 | previsões [0.11 0.86 0.88 0.18]


**Conclusão (XOR):** loss = 0.25 significa que a rede responde 0.5 a tudo (erro de 0.5² em cada caso). Mesmo na versão base, a loss fica nesse patamar ~1000 épocas antes de cair; a rede acerta os 4 casos na época 2207.

| Variação | O que observas? |
|---|---|
| Tira a SigmoidActivation do meio | Fica presa em 0.5 para tudo (accuracy 0.5), mesmo com 30000 épocas. Sem ativação, duas camadas densas seguidas são uma só transformação linear (X·W1·W2 = X·W'), e a rede fica igual a uma regressão logística: só separa com uma reta, e o XOR não se separa com uma reta. |
| learning_rate=0.01 | Ao fim de 5000 épocas ainda responde 0.5, mas não está presa, é só lenta: o passo é 50× mais pequeno e precisa de ~50× mais épocas (accuracy 1 na época 110165, contra 2207 com 0.5). |
| np.random.seed(0) | Com outros pesos iniciais, fica muito mais tempo no patamar de 0.25 e só aprende por volta da época 10962. O resultado (e a velocidade) depende da inicialização dos pesos. |

## EX 14 - CategoricalCrossEntropy


In [7]:
from si.neural_networks.losses import CategoricalCrossEntropy

cce = CategoricalCrossEntropy()

# gradient check, com 5 amostras e 3 classes
np.random.seed(0)
y_true = np.eye(3)[np.random.randint(0, 3, size=5)]   # one-hot: cada linha tem um 1 na classe certa
y_pred = np.random.uniform(0.05, 0.95, size=(5, 3))
y_pred = y_pred / y_pred.sum(axis=1, keepdims=True)   # cada linha soma 1, como à saída de uma softmax
print("one-hot:\n", y_true)
print(f"loss = {cce.loss(y_true, y_pred):.4f} | diferença máx. = "
      f"{np.abs(cce.derivative(y_true, y_pred) - derivada_numerica(cce, y_true, y_pred)).max():.1e}")

# só conta a probabilidade dada à classe certa (aqui a classe do meio)
y = np.array([[0.0, 1.0, 0.0]])
print("p classe certa = 0.8  ->", round(cce.loss(y, np.array([[0.1, 0.8, 0.1]])), 3))
print("p classe certa = 0.1  ->", round(cce.loss(y, np.array([[0.8, 0.1, 0.1]])), 3))
print("p classe certa = 0    ->", round(cce.loss(y, np.array([[1.0, 0.0, 0.0]])), 3), "(sem o clip dava inf)")

one-hot:
 [[1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 1. 0.]]
loss = 6.3101 | diferença máx. = 4.7e-10
p classe certa = 0.8  -> 0.223
p classe certa = 0.1  -> 2.303
p classe certa = 0    -> 34.539 (sem o clip dava inf)


**Conclusão (Exercício 14):**
- A `derivative` bate certo com a derivada numérica (diferença ~1e-10): −y/p é a derivada da loss.
- Com y em one-hot, só conta a classe certa: a loss de cada amostra é −log(p da classe certa). p = 0.8 dá 0.223; p = 0.1 dá 2.303.
- É a generalização da BCE: com 2 classes, o one-hot é [y, 1 − y] e a fórmula fica igual à da BCE.
- O `np.clip` evita log(0) e a divisão por 0: com p = 0 na classe certa, a loss fica 34.5 em vez de inf.

## EX15 - Adam

In [8]:
from si.neural_networks.optimizers import SGD, Adam

# o 1.º passo do Adam é ~learning_rate para TODOS os parâmetros, seja qual for o tamanho do gradiente
# (no SGD o passo é proporcional ao gradiente: 0.001 quase não mexe, 1000 dá um salto enorme)
w = np.array([[1.0, 1.0, 1.0]])
grad = np.array([[0.001, 1.0, 1000.0]])
adam = Adam(learning_rate=0.001)
print("Adam:", adam.update(w, grad))
print("SGD: ", SGD(learning_rate=0.001).update(w, grad))

# correção do viés: depois de 1 passo, m = 0.1 * gradiente, mas m_hat = gradiente
print("t =", adam.t, "| m =", adam.m, "| m_hat =", adam.m / (1 - adam.beta_1 ** adam.t))

# XOR (rede do slide 14): SGD com learning_rate 0.5 contra Adam com learning_rate 0.01
for nome, optimizer, lr in [("SGD ", SGD, 0.5), ("Adam", Adam, 0.01)]:
    for seed in [1, 0]:
        np.random.seed(seed)
        net = NeuralNetwork(epochs=15000, batch_size=4, optimizer=optimizer, learning_rate=lr,
                            loss=MeanSquaredError, metric=accuracy)
        net.add(DenseLayer(4, (2,)))
        net.add(SigmoidActivation())
        net.add(DenseLayer(1))
        net.add(SigmoidActivation())
        net.fit(ds)
        print(f"{nome} (lr = {lr}), seed {seed}: accuracy 1 na época {epoca_accuracy_1(net)} "
              f"| previsões {net.predict(ds).round(2).ravel()}")

Adam: [[0.99900001 0.999      0.999     ]]
SGD:  [[0.999999 0.999    0.      ]]
t = 1 | m = [[1.e-04 1.e-01 1.e+02]] | m_hat = [[1.e-03 1.e+00 1.e+03]]
SGD  (lr = 0.5), seed 1: accuracy 1 na época 2207 | previsões [0.02 0.98 0.98 0.03]
SGD  (lr = 0.5), seed 0: accuracy 1 na época 10962 | previsões [0.04 0.96 0.96 0.04]
Adam (lr = 0.01), seed 1: accuracy 1 na época 120 | previsões [0. 1. 1. 0.]
Adam (lr = 0.01), seed 0: accuracy 1 na época 273 | previsões [0. 1. 1. 0.]


**Conclusão (Exercício 15):**
- No 1.º passo, o Adam mexe ~0.001 (o learning_rate) em todos os parâmetros, com gradientes de 0.001, 1 ou 1000 (o 0.99900001 é o efeito do epsilon). No SGD, o passo é proporcional ao gradiente: o 1.º peso quase não mexe e o 3.º salta de 1 para 0. O Adam divide pelo tamanho típico do gradiente (√v), por isso cada parâmetro tem o seu passo.
- Correção do viés: como m começa a 0, no 1.º passo m = 0.1·gradiente; m_hat = m / (1 − β1^t) devolve o gradiente certo.
- No XOR, o Adam (lr 0.01) acerta os 4 casos na época 120 (seed 1) e 273 (seed 0), contra 2207 e 10962 do SGD (lr 0.5). No patamar de loss 0.25 os gradientes são muito pequenos: o SGD dá passos pequenos e demora a sair; o Adam normaliza o passo e sai do patamar muito mais depressa.
- O `t` conta atualizações (uma por mini-batch), e não épocas.

## Ex16 - Rede classificação binária com k-fold

In [11]:
from si.model_selection.cross_validation import k_fold_cross_validation
from si.neural_networks.activation import ReLUActivation
from si.neural_networks.losses import BinaryCrossEntropy

# dataset: 500 amostras, 32 features, classificação binária
# a classe segue uma regra escondida (uma combinação linear das features, com algum ruído),
# para haver alguma coisa para a rede aprender
np.random.seed(42)
X = np.random.randn(500, 32)
pesos_escondidos = np.random.randn(32)
# o np.errstate esconde avisos falsos do numpy no Mac (biblioteca Accelerate); os valores estão certos
with np.errstate(all='ignore'):
    y = (X @ pesos_escondidos + 0.5 * np.random.randn(500) > 0).astype(float)
dataset = Dataset(X, y)
print("dataset:", dataset.shape(), "| proporção da classe 1:", round(float(y.mean()), 3))

# rede: as camadas densas reduzem as unidades para metade (32 -> 16 -> 8), e a de saída tem 1
net = NeuralNetwork(epochs=100, batch_size=16, optimizer=SGD, learning_rate=0.01,
                    loss=BinaryCrossEntropy, metric=accuracy)
net.add(DenseLayer(16, (32,)))
net.add(ReLUActivation())
net.add(DenseLayer(8))
net.add(ReLUActivation())
net.add(DenseLayer(1))
net.add(SigmoidActivation())

for layer in net.layers:
    print(f"{layer.layer_name():18} input {layer.input_shape()} -> output {layer.output_shape()} | parâmetros: {layer.parameters()}")

# 5-fold cross validation: 5 treinos, cada um com 400 amostras de treino e 100 de teste
scores = k_fold_cross_validation(net, dataset, cv=5, seed=42)
print("accuracy em cada fold:", [round(float(s), 3) for s in scores])
print(f"accuracy (5-fold CV): {np.mean(scores):.3f} ± {np.std(scores):.3f}")

# para comparar: a mesma rede treinada com os dados todos, avaliada nos próprios dados de treino
net.fit(dataset)
print(f"accuracy no próprio treino: {net.score(dataset):.3f}")

dataset: (500, 32) | proporção da classe 1: 0.522
DenseLayer         input (32,) -> output (16,) | parâmetros: 528
ReLUActivation     input (16,) -> output (16,) | parâmetros: 0
DenseLayer         input (16,) -> output (8,) | parâmetros: 136
ReLUActivation     input (8,) -> output (8,) | parâmetros: 0
DenseLayer         input (8,) -> output (1,) | parâmetros: 9
SigmoidActivation  input (1,) -> output (1,) | parâmetros: 0
accuracy em cada fold: [0.9, 0.9, 0.92, 0.91, 0.91]
accuracy (5-fold CV): 0.908 ± 0.007
accuracy no próprio treino: 1.000


**Conclusão (Exercício 16):**
- Arquitetura: 32 → 16 → 8 → 1. Cada camada densa reduz as unidades para metade, exceto a de saída; ReLU nas escondidas e sigmoide na saída (probabilidade da classe 1). Parâmetros: 528 + 136 + 9 = 673.
- Dados: 500 amostras sintéticas com 32 features; a classe segue uma regra escondida com ruído. Com rótulos totalmente aleatórios, qualquer modelo ficaria em ~0.5, porque não haveria nada para aprender.
- 5-fold CV: accuracy 0.908 ± 0.007. Os 5 folds dão valores muito parecidos, por isso a estimativa é estável.
- No próprio treino, a accuracy é 1.000: a rede decorou os dados de treino (overfitting). A diferença entre 1.000 e 0.908 mostra porque se avalia o modelo em dados que ele não viu.
- Em cada fold, a `k_fold_cross_validation` treina uma cópia do modelo por treinar (`copy.deepcopy`). Sem isso, a rede continuaria a treinar com os pesos do fold anterior, que já tinham visto as amostras de teste.